# B2.5 · Feasibility filtering, reachability and dead code

**Function B — Application Security with an AI SDLC → The AI SDLC: an Agentic AppSec Pipeline, Before and After Deploy**  ·  *AI for Security*

Builds on **[B2.4 · Deduplication and contextual verification](https://spbreed.github.io/cyber-commons/lessons/B2.4.html)**.

| | |
|---|---|
| Tools used | CodeQL, tree-sitter, GLM-4.6, Claude Sonnet 5 |

## What this lesson is

**What it covers.** Build a call graph from entry points and partition findings into reachable, unreachable and unknown.

**Why a security engineer needs it.** A finding in dead code costs the same to triage as one on the login path. The control it builds is: stage 10: decide whether an external caller can actually reach the sink before anyone is paged.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

The finding is real. The code is dead. Reachability is the difference between a queue an engineer works and a queue an engineer learns to ignore, and it is the single largest false-positive killer in the pipeline.

> **At CyberTravels.** The finding is real and the code is unreachable from any traveller input. Reachability is what stops CyberTravels' queue becoming something engineers learn to ignore.

## 2 · The framework

```
   is there a path from untrusted input to this line?

   HTTP handler --> parse() --> validate() --> build_query() --> DB
                                                    ^
                                              the finding

   reachable   -> a finding
   unreachable -> a note

   the largest single false-positive killer in the pipeline
```

**Stage 10 — Feasibility filtering.** The last stage of Phase 3, and the one
that decides whether anyone gets paged.

A verified finding is a real bug in the code. It is not necessarily a real risk,
because the code may be unreachable: dead code, a test fixture, an internal
function no external caller can drive, a branch behind a feature flag that has
been off for two years.

Triaging an unreachable finding costs exactly as much as triaging one on the
login path, and there are usually far more of them. So this stage partitions
findings into three buckets — and the third bucket is the honest one:

- **reachable** — a path exists from an untrusted entry point to the sink,
- **unreachable** — no path exists,
- **unknown** — the analysis cannot decide, usually because of dynamic dispatch,
  reflection, or a framework that wires callers at runtime.

Reporting `unknown` as `unreachable` is how a pipeline quietly drops real bugs.

### Dead code, and the two different claims a finding makes

The largest single class in that unreachable bucket is **dead code**, and it is
worth being precise about what is wrong with such a finding, because teams act
on the wrong half.

The finding is a **true positive about the code**. The concatenation is there,
the sink is real, and any reviewer who opens the file will agree. It is a
**false positive about the risk**, because nothing untrusted reaches it. Two
different claims, and only the second one is wrong.

That distinction decides the response. The instinct is to **suppress** — it
empties the queue, and it is the wrong fix for a reason that only appears
months later: a suppression is keyed to a file, a line and a rule, and *none of
those change when somebody wires the function back up*. The code becomes live,
the finding does not come back, and the entry that was hiding a false positive
is now hiding a real one.

**Attack surface reduction — ASR, which here means deleting the code — closes
both halves at once.** The finding goes because the code is gone, and so does
the latent risk of it being reconnected. It is the only response to a dead-code
finding that cannot rot, and the pleasant surprise is that the security queue
turns out to be the cheapest to-delete list in the building: already
enumerated, already ranked by what each line would cost if it ever became
reachable again.

One caution that does most of the work: "unreachable" and "dead" are not
synonyms. A test fixture and a feature flag that has been off for two years are
both unreachable *under a condition*, and both become reachable the day
somebody changes one line. Only code with no caller anywhere is a deletion
candidate.

> **Where you are in the pipeline.**
>
> ```
> [Ingestion & Mapping] ──> [Threat Modelling] ──> [Discovery]
>          └─ stages 1-4         └─ stages 5-6        └─ stages 7-10
>                    ──> [Dynamic Validation] ──> [Reporting]
>                              └─ stages 11-14        └─ stage 15
> ```

## 3 · Where it breaks — collapsing `unknown` into `unreachable`

The tempting simplification. It makes the queue shorter and it is how real bugs get dropped, because dynamic dispatch is exactly where framework-wired handlers live.

## 4 · Phase 3 as a skill — and the counts that police it

Stages 7 to 10 only ever *shrink* the list. That is a property worth enforcing rather than trusting, so the skill's contract carries a `counts` object and the rule that it must never increase.

A pipeline whose `verified` count exceeds its `deduped` count has invented findings somewhere after the audit stage — and that is far easier to do by accident than it sounds, because a verification step that expands one finding per code path looks perfectly reasonable from the inside.

### The skill — [`skills/appsec/appsec-vuln-audit/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/appsec/appsec-vuln-audit/SKILL.md)

```yaml
name: appsec-vuln-audit
description: >-
  Audit code for vulnerabilities against a threat model, then deduplicate,
  verify in context, and filter to what is actually reachable. Use when asked
  to review code for security bugs, run or interpret SAST, check whether a
  finding is a false positive, or reduce a noisy findings list to the ones
  worth a human's time.
allowed-tools: Read, Grep, Glob, Bash
```

# AppSec pipeline · Phase 3 — Analysis and filtering

Covers **stages 7–10**. This is where findings are produced — and, more
importantly, where most of them are thrown away.

The hard problem in application security is not finding candidate defects. It
is that a scanner emits hundreds and a human can act on ten. Every stage after
7 exists to shrink the list without losing the true positives.

## When to use this

When you have a threat model and a budget, or when handed a raw findings file
that nobody trusts. Stages 8–10 work on any findings list, including one from a
third-party scanner.

## Inputs

| Input | Required | Notes |
|---|---|---|
| `threat_model` + `plan.selected` | preferred | from appsec-threat-model |
| Source worktree | yes | verification needs the code, not just the finding |
| Existing findings | optional | run stages 8–10 alone to clean a noisy list |

## Procedure

**Stage 7 — Vulnerability auditing.** For each selected threat, examine the
path from entry to sink and decide whether the weakness is actually present.
Record for each finding: `cwe`, `file`, `line`, `unit`, the **evidence** (the
specific expression that is unsafe), and the **sanitiser** you looked for and
did not find. A finding that cannot name what was missing is a guess.

Three generations of analysis, and they are complementary, not competing:
grep-class pattern matching (fast, no dataflow), taint analysis (dataflow, no
semantics), and model-assisted review (semantics, no guarantees). Use the
cheapest one that can answer the question, and never let the third overrule the
second on a question of reachability — the model does not execute the program.

**Stage 8 — Deduplication.** The same defect appears many times: once per
scanner, once per path, once per call site. Collapse on the **defect identity**
— `(cwe, file, unit, sink_expression)` — not on the message text. Keep the
count: `occurrences` is signal about how exposed the defect is.

Match paths by parent directory plus filename tail. Deduplicating on a bare
basename silently merges two different files and loses a real finding.

**Stage 9 — Contextual verification.** For each surviving finding, look at the
surrounding code for the thing that makes it not-a-bug: a validator upstream, a
framework escaping the parameter, a type that cannot hold the payload, a caller
that only ever passes a constant. Record the verdict and the reason:
`confirmed`, `mitigated_by <what>`, or `needs_human`.

`needs_human` is a legitimate verdict and must stay available. A pipeline that
must decide will decide wrongly under uncertainty.

**Stage 10 — Feasibility filtering.** Drop what an attacker cannot actually
reach: code behind a feature flag that is off, an admin-only path in a service
with no admin, a sink whose input is fully constant. Record *why* each drop was
made, because the next scan will rediscover it and the reason is what stops
that work being repeated.

## Output contract

```json
{
  "findings": [
    {"id": "str", "cwe": "CWE-89", "file": "str", "line": 0, "unit": "str",
     "evidence": "str", "missing_control": "str",
     "occurrences": 1, "verdict": "confirmed|mitigated|needs_human",
     "verdict_reason": "str", "feasible": true, "confidence": 0.0}
  ],
  "dropped": [{"id": "str", "stage": 8, "why": "str"}],
  "counts": {"raw": 0, "deduped": 0, "verified": 0, "feasible": 0}
}
```

`counts` must be monotonically non-increasing across the four stages. If it is
not, the pipeline invented findings after the audit stage — stop and report.

## Failure modes

- **Confusing conformance with accuracy.** Output that matches this schema
  perfectly can still be entirely wrong. Schema validity is close to free;
  correctness is the expensive part. Never report conformance as a quality
  metric.
- **Dropping silently.** Every drop needs a stage and a reason.
- **Letting a model overrule dataflow on reachability.** It may propose a path;
  it may not confirm one.
- **Suppressing `needs_human` to look decisive.** Uncertainty that is hidden
  becomes someone's incident.

## Handoff

Feasible, confirmed findings go to **appsec-exploit-validate** for proof.
Everything else goes to **appsec-triage-report** with its verdict intact.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/appsec/appsec-vuln-audit/scripts/appsec_vuln_audit.py
SCRIPT = "skills/appsec/appsec-vuln-audit/scripts/appsec_vuln_audit.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; `sparse-checkout set skills` then materialises only what runs.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set", "skills"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## 5 · Where it breaks — deduplicating on the wrong key

The skill says to collapse on the **defect identity**, `(cwe, file, unit, sink_expression)`, and never on the message text. Here is why that sentence is in the procedure.

## 6 · The same failure, from a real model

Everything above is constructed. Here is the identical failure produced by an actual open-weight model — **Moonlight-16B-A3B**, Moonshot AI's MoE from the Kimi team — run on a Kaggle CPU kernel against this skill's output contract.

It was given the contract and two vulnerable functions: an `open()` on a caller-supplied path, and an `os.system()` on a caller-supplied argument. Its answer is reproduced verbatim below ([full run](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/kimi/moonlight-16b-completion-prompt.txt)).

## 7 · Read that output again

It passes the contract with zero problems, and almost nothing in it is true.

## 8 · Dead code, suppression and ASR, as a skill

The same eight findings from CyberTravels' booking service, partitioned into the three buckets — then the part that decides what to do with the unreachable ones.

It compares the three available responses to a single dead `os.system` finding on three axes: does the finding go, does the risk go, and does the decision **rot**. Then it applies one commit six weeks later that re-imports the dead function, and shows which of the three responses is still protecting anything.

### The skill — [`skills/appsec/dead-code-attack-surface/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/appsec/dead-code-attack-surface/SKILL.md)

```yaml
name: dead-code-attack-surface
description: >-
  Separate findings that are false positives about risk from findings that are
  false positives about code, and decide between suppressing a dead-code
  finding and deleting the code. Use when a queue is dominated by unreachable
  findings, when suppressions are being added in bulk, or when asked why a
  known-dead vulnerability came back live.
allowed-tools: Read, Grep, Glob
```

# Dead code is a true positive about the code and a false positive about the risk

Both halves of that sentence matter and teams usually act on only one.

The finding is **correct**: the concatenation is there, the sink is real, a
reviewer who opens the file will agree. What is wrong is the implied
consequence, because nothing untrusted can reach it. Triaging it costs exactly
as much as triaging one on the login path, and there are usually far more of
them — so a queue that does not separate the two is a queue engineers learn to
ignore, which costs you the reachable ones too.

The response almost everyone reaches for is a **suppression**. It empties the
queue and it is the wrong fix, for a reason that only shows up months later: a
suppression is keyed to a file, a line and a rule, and **none of those change
when somebody wires the function back up**. The code becomes reachable, the
finding does not come back, and the suppression is now hiding a live
vulnerability.

**Attack surface reduction (ASR) — deleting the code — resolves both halves at
once.** The finding goes because the code is gone, and the latent risk goes
with it. It is the only response to a dead-code finding that cannot rot.

## When to use this

When reachability analysis produces a large unreachable bucket, before any bulk
suppression, and during any dead-code or deprecation sweep — the security queue
is the cheapest available list of which dead code to delete first.

## Procedure

**1 — Partition by reachability, in three buckets, never two.** Reachable,
unreachable, and unknown. Collapsing unknown into unreachable is how real bugs
are dropped, and dynamic dispatch is exactly where framework-wired handlers
live.

**2 — For the unreachable bucket, ask why it is unreachable.** Dead code that
nothing calls, a test fixture, a feature flag that has been off for two years,
or an internal function with no external driver. Only the first is a deletion
candidate; the others are reachable under a condition you have not enumerated.

**3 — Rank deletion candidates by what they would cost if reachable.** The
dead-code queue is a to-delete list, and the security severity is the ordering
you already have. A dead `os.system` on a caller-supplied path is a better
first deletion than a dead string formatter.

**4 — Delete, and re-run the scan to prove the finding is gone.** The proof is
that the finding disappears without a suppression entry. If it needed one, the
code was not dead.

**5 — For anything you suppress instead, bind the suppression to reachability,
not to the line.** A suppression that does not expire, and does not re-open when
the call graph changes, is a permanent decision made on temporary evidence.

## Output contract

```json
{
  "buckets": {"reachable": 0, "unreachable": 0, "unknown": 0},
  "queue": {"before": 0, "after_reachability": 0},
  "responses": [{"finding": "str", "action": "suppress|wont-fix|delete",
                 "finding_cleared": true, "risk_cleared": true, "rots": true}],
  "asr": {"deleted": 0, "surface_removed": ["str"]},
  "resurrected": 0
}
```

`resurrected` counts findings whose code became reachable again while a
suppression was still in place. It should be zero, and the only way to keep it
zero is to have deleted the code or expired the suppression.

## Failure modes

- **Suppressing in bulk to clear the queue.** It works, and it converts a
  visible false positive into an invisible true one.
- **Collapsing `unknown` into `unreachable`.** The bucket that gets dropped is
  the one that contains the framework-wired handlers.
- **Deleting on the scanner's word alone.** Reachability from *untrusted* entry
  points is not reachability from anywhere; check the call graph before the
  delete, not after.
- **Treating ASR as a security project.** It is a deletion, it belongs in
  ordinary maintenance, and framing it as a programme is how it never happens.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/appsec/dead-code-attack-surface/scripts/dead_code_attack_surface.py
SCRIPT = "skills/appsec/dead-code-attack-surface/scripts/dead_code_attack_surface.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; `sparse-checkout set skills` then materialises only what runs.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set", "skills"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## 9 · The commit six weeks later

That last block is the argument, and it is worth restating plainly because it
is the part that gets waved through in a triage meeting.

A suppression matched on `(file, line, rule)`. The re-enabling commit changed
none of the three — it added an import in a different file. So the suppression
still matches, the scanner still stays quiet, and a `sev 9` command injection is
now reachable from the vendor webhook with no finding attached to it.

The deletion cannot fail that way. The import does not resolve, the build breaks
at the moment somebody tries to bring the code back, and the failure is loud and
immediate rather than silent and eighteen months old.

If you must suppress, bind the suppression to **reachability** rather than to a
line, and give it an expiry. A suppression with neither is a permanent decision
recorded against temporary evidence.

## What you just proved

The call graph identifies three entry points, one of which uses dynamic dispatch. `load_report` is reachable, `debug_dump` and `legacy_export` are unknown rather than unreachable because runtime handler resolution cannot be ruled out. Two-bucket filtering silently drops both, and the three-bucket routing sends the unknowns to Phase 4 instead of paging or discarding them. On the eight-finding queue, reachability takes 8 down to 2, and three of the remainder are dead with no caller anywhere — a deletion rather than a triage. Suppressing that dead `os.system` and marking it won't-fix both clear the finding and neither clears the risk: one commit six weeks later re-imports the function, the suppression still matches on file, line and rule, and the finding never comes back. Deleting it breaks that commit's build instead.

## Your turn

Two counts, and the second is the uncomfortable one. Count how many `unknown` cases your own reachability analysis produces and find out what your tooling does with them — if it reports them as clean, the number of real bugs you are dropping is the size of that bucket. Then open your suppression file and find the oldest entry. Check whether the code it covers is still unreachable, and whether anything in your pipeline would have told you if it stopped being.

---

**Next → [B2.6 · Sandbox replication](https://spbreed.github.io/cyber-commons/lessons/B2.6.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.5.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.5.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*